In [1]:
from xbbg import blp, Backend
import pandas as pd
from pandas.tseries.holiday import USFederalHolidayCalendar
from pandas.tseries.offsets import CustomBusinessDay
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo

# https://xbbg.org/
# https://pypi.org/project/xbbg/

In [2]:
tickers = ["AAPL US Equity", "6758 JP Equity"]
fields = ["name", "crncy", "px_last", "px_open", "volume"]

df_bdp = blp.bdp(tickers,fields,backend=Backend.PANDAS)
display(df_bdp.head())

df_bdp = df_bdp.pivot(index="ticker", columns="field", values="value").reset_index().rename_axis(None, axis=1)
df_bdp = df_bdp[fields]
display(df_bdp)

,ticker,field,value
0,AAPL US Equity,name,APPLE INC
1,AAPL US Equity,crncy,USD
2,AAPL US Equity,px_last,306.24
3,AAPL US Equity,px_open,307.75
4,AAPL US Equity,volume,5394088.0


,name,crncy,px_last,px_open,volume
0,SONY GROUP CORP,JPY,3764.0,3707.0,16907600.0
1,APPLE INC,USD,306.24,307.75,5394088.0


In [3]:
tickers = ["AAPL US Equity", "6758 JP Equity"]
fields = ["px_last","px_open","volume"]

df_bdh = blp.bdh(tickers,fields,"2026-05-01","2026-05-15",backend=Backend.PANDAS,Per="W")
display(df_bdh.head(4))


df_bdh = (df_bdh
    .assign(column_name=lambda x: x["ticker"].str.split().str[0] + "_" + x["field"])
    .pivot(index="date", columns="column_name", values="value")
    .reset_index()
    .rename_axis(None, axis=1)
)


tickers_short = [t.split()[0] for t in tickers]

ordered_cols = ["date"] + [
    f"{t}_{f}" for t in tickers_short for f in fields
]

df_bdh = df_bdh[ordered_cols]
display(df_bdh.head(3))

,ticker,date,field,value
0,AAPL US Equity,2026-05-01,px_last,2.801400e+02
1,AAPL US Equity,2026-05-01,px_open,2.660900e+02
2,AAPL US Equity,2026-05-01,volume,2.832972e+08
3,AAPL US Equity,2026-05-08,px_last,2.933200e+02


,date,AAPL_px_last,AAPL_px_open,AAPL_volume,6758_px_last,6758_px_open,6758_volume
0,2026-05-01,280.14,266.090,283297243.0,3127.0,3193.0,72714900.0
1,2026-05-08,293.32,279.655,252233246.0,3114.0,3124.0,97760000.0
2,2026-05-15,300.23,291.979,230867432.0,3576.0,3352.0,219785800.0


In [4]:
df = blp.bds(
        "AAPL US Equity",
        "DVD_HIST_ALL",
        backend=Backend.PANDAS
    )

df.head()

,ticker,field,Declared Date,Ex-Date,Record Date,Payable Date,Dividend Amount,Dividend Frequency,Dividend Type
0,AAPL US Equity,DVD_HIST_ALL,2026-07-30,2026-08-10,2026-08-10,2026-08-13,0.27,Quarter,Regular Cash
1,AAPL US Equity,DVD_HIST_ALL,2026-04-30,2026-05-11,2026-05-11,2026-05-14,0.27,Quarter,Regular Cash
2,AAPL US Equity,DVD_HIST_ALL,2026-01-29,2026-02-09,2026-02-09,2026-02-12,0.26,Quarter,Regular Cash
3,AAPL US Equity,DVD_HIST_ALL,2025-10-30,2025-11-10,2025-11-10,2025-11-13,0.26,Quarter,Regular Cash
4,AAPL US Equity,DVD_HIST_ALL,2025-07-31,2025-08-11,2025-08-11,2025-08-14,0.26,Quarter,Regular Cash


In [5]:
# TZ = "America/New_York"
TZ = "Asia/Tokyo"

prev_date = (
    datetime.now(ZoneInfo(TZ)) - timedelta(days=1)
).strftime("%Y-%m-%d")

bars_1m = blp.bdib(
    ticker="ES1 A:00_0_R COMB Index",
    dt=prev_date,
    typ="TRADE",
    interval=1,
    backend=Backend.PANDAS,
    request_tz=TZ,
    output_tz=TZ,
)

print(len(bars_1m))
display(bars_1m.head(3))
display(bars_1m.tail(3))

1020


,ticker,time,open,high,low,close,volume,numEvents,value
0,ES1 A:00_0_R COMB Index,2026-08-10 07:00:00+09:00,7776.75,7779.25,7773.50,7775.5,1021.0,460,7939542.00
1,ES1 A:00_0_R COMB Index,2026-08-10 07:01:00+09:00,7775.25,7777.00,7773.75,7774.5,205.0,115,1594021.25
2,ES1 A:00_0_R COMB Index,2026-08-10 07:02:00+09:00,7774.75,7775.75,7774.00,7774.5,290.0,161,2254710.25


,ticker,time,open,high,low,close,volume,numEvents,value
1017,ES1 A:00_0_R COMB Index,2026-08-10 23:57:00+09:00,7786.25,7786.25,7784.25,7785.25,2747.0,981,21386146.0
1018,ES1 A:00_0_R COMB Index,2026-08-10 23:58:00+09:00,7785.00,7786.00,7784.00,7784.25,2303.0,845,17928682.0
1019,ES1 A:00_0_R COMB Index,2026-08-10 23:59:00+09:00,7784.50,7785.00,7781.25,7782.00,3668.0,1286,28549378.0


In [6]:
def get_intraday_bars(
    tickers,
    interval=1,
    tz="Asia/Tokyo",
):
    # Today and previous calendar day in JST
    today = datetime.now(ZoneInfo(tz)).date()

    dates = [
        (today - timedelta(days=1)).strftime("%Y-%m-%d"),
        today.strftime("%Y-%m-%d"),
    ]

    dfs = []

    for dt in dates:
        for ticker in tickers:

            df = blp.bdib(
                ticker=ticker,
                dt=dt,
                typ="TRADE",
                interval=interval,
                backend=Backend.PANDAS,
                request_tz=tz,
                output_tz=tz,
            )

            if df.empty:
                continue

            df = df.copy()
            df["date_jst"] = dt

            dfs.append(df)

    if not dfs:
        return pd.DataFrame()

    return (
        pd.concat(dfs, ignore_index=True)
        .sort_values(["time", "ticker"])
        .reset_index(drop=True)
    )


bars_1m = get_intraday_bars(
    tickers=[
        "ES1 A:00_0_R COMB Index",
        "TP1 A:00_0_R COMB Index",
    ],
)

print(len(bars_1m))
display(bars_1m.head(3))
display(bars_1m.tail(3))

4144


,ticker,time,open,high,low,close,volume,numEvents,value,date_jst
0,ES1 A:00_0_R COMB Index,2026-08-10 07:00:00+09:00,7776.75,7779.25,7773.50,7775.5,1021.0,460,7939542.00,2026-08-10
1,ES1 A:00_0_R COMB Index,2026-08-10 07:01:00+09:00,7775.25,7777.00,7773.75,7774.5,205.0,115,1594021.25,2026-08-10
2,ES1 A:00_0_R COMB Index,2026-08-10 07:02:00+09:00,7774.75,7775.75,7774.00,7774.5,290.0,161,2254710.25,2026-08-10


,ticker,time,open,high,low,close,volume,numEvents,value,date_jst
4141,TP1 A:00_0_R COMB Index,2026-08-11 23:01:00+09:00,4139.50,4140.00,4139.00,4139.50,18.0,17,74510.5,2026-08-11
4142,ES1 A:00_0_R COMB Index,2026-08-11 23:02:00+09:00,7786.25,7786.25,7784.25,7784.25,2171.0,745,16901640.0,2026-08-11
4143,TP1 A:00_0_R COMB Index,2026-08-11 23:02:00+09:00,4139.00,4139.50,4138.00,4138.50,28.0,13,115884.0,2026-08-11


In [7]:
TZ = "Asia/Tokyo"

prev_date = (
    datetime.now(ZoneInfo(TZ)) - timedelta(days=1)
).strftime("%Y-%m-%d")

ticks = blp.bdtick(
    ticker="ES1 Index",
    start_datetime=f"{prev_date} 09:30:00",
    end_datetime=f"{prev_date} 09:31:00",
    event_types=["TRADE", "BID"],
    backend=Backend.PANDAS,
    request_tz=TZ,
    output_tz=TZ,
)
print(prev_date)
display(ticks.head(3))
display(ticks.tail(3))

2026-08-10


,ticker,time,type,value,size
0,ES1 Index,2026-08-10 09:30:00.003000+09:00,BID,7770.75,7
1,ES1 Index,2026-08-10 09:30:00.004000+09:00,BID,7770.75,6
2,ES1 Index,2026-08-10 09:30:00.004000+09:00,BID,7770.75,4


,ticker,time,type,value,size
1142,ES1 Index,2026-08-10 09:31:00.860000+09:00,BID,7772.5,8
1143,ES1 Index,2026-08-10 09:31:00.860000+09:00,BID,7772.5,10
1144,ES1 Index,2026-08-10 09:31:00.860000+09:00,BID,7772.5,11
